# BarberStudio — Entrenamiento del segmentador de cabello (Colab)

Este notebook entrena `yolov8n-seg` con fine-tuning sobre **Figaro1k + CelebAMask-HQ** en una GPU T4 gratuita (~1 hora para 60 épocas). Al final descargas `best.pt`, el modelo especializado en cabello.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU T4**.

Flujo: GPU ok → instalar → datasets → convertir → entrenar → descargar.

## 1) Verificar GPU

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2) Instalar Ultralytics y clonar el repo
Usamos los mismos scripts del repo (conversión y entrenamiento) para que Colab y local sean idénticos.

In [ ]:
!pip install -q ultralytics
!git clone https://github.com/kaloslazo/barber-studio.git
%cd /content/barber-studio/backend

## 3) Poner los datasets en `data/`

**Opción A (recomendada):** crea en tu Google Drive la carpeta `barberstudio` y sube ahí los dos zips:
- `jy3cd-osfstorage-archive.zip` (Figaro1k, 114 MB)
- `CelebAMask-HQ.zip` (3.15 GB)

Al montar Drive, los copiamos al disco de Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIPS = '/content/drive/MyDrive/barberstudio'
!mkdir -p ../data
!cp "{ZIPS}/jy3cd-osfstorage-archive.zip" ../data/
!cp "{ZIPS}/CelebAMask-HQ.zip" ../data/

**Opción B (si no usas Drive):** súbelos directo desde tu Mac (más lento). Descomenta y ejecuta:

In [ ]:
# from google.colab import files
# import shutil
# uploaded = files.upload()
# for name in uploaded:
#     shutil.move(name, f'../data/{name}')

## 4) Descomprimir (mismo layout que en local)

In [ ]:
!unzip -q -o ../data/jy3cd-osfstorage-archive.zip -d ../data/figaro1k-raw
!unzip -q -o ../data/figaro1k-raw/Figaro1K/Figaro1k.zip -d ../data/figaro1k
!unzip -q -o ../data/CelebAMask-HQ.zip -d ../data/celebamask-raw
!echo datasets listos

## 5) Convertir al formato YOLO-seg (~5 min)
Genera `data/yolo-hair/` con ~9,200 muestras (train/val) + `hair.yaml`.

In [ ]:
!python training/prepare_hair_dataset.py

## 6) Entrenar — 60 épocas (~1 h en T4)
Parte de los pesos COCO (`yolov8n-seg.pt`) y los especializa en cabello. Las curvas de loss/mAP quedan en `training/runs/hair-v1-colab/` (material directo para el informe LaTeX).

In [ ]:
!python training/train_hair.py --epochs 60 --batch 32 --device 0 --name hair-v1-colab

## 7) Ver resultados (curvas y predicciones del set de validación)

In [ ]:
from IPython.display import Image, display
display(Image(filename='training/runs/hair-v1-colab/results.png'))
display(Image(filename='training/runs/hair-v1-colab/val_batch0_pred.jpg'))

## 8) Descargar `best.pt`
Guárdalo en tu Mac en `barber-studio/backend/models/hair_best.pt` (esa carpeta está gitignored: los pesos NO van al repo). Con eso la app local puede inferir en CPU.

In [ ]:
from google.colab import files
files.download('training/runs/hair-v1-colab/weights/best.pt')

## Extra: si Colab te corta la sesión
Vuelve a correr las celdas 1-5 (datasets desde Drive) y retoma el entrenamiento donde quedó:
```
!yolo segment train model=training/runs/hair-v1-colab/weights/last.pt resume=True
```